In [158]:
import json
import pandas as pd

In [159]:


sheet_id = "1_3BJkGRZHEvq0XkfJGOB3lCe8rJgeGG5JW7vnY_uxxw"          # ← replace
sheet_name = "Sheet1"                        # or leave blank for first sheet

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
df = pd.read_csv(url)




In [160]:
required_columns = ['linkedinUrl', 'content', 'contentAttributes', 'author', 'postedAt']

linkedIn_post = df[required_columns]
linkedIn_post.rename(columns={"linkedinUrl": "post_url", "content": "post_content"}, inplace = True)

C:\Users\User\AppData\Local\Temp\ipykernel_6248\1446819216.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linkedIn_post.rename(columns={"linkedinUrl": "post_url", "content": "post_content"}, inplace = True)


In [161]:

def fix_email_extraction(row):
    """Custom extraction based on your actual data structure"""
    emails = []
    
    try:
        attr = row['contentAttributes']
        
        # Based on your earlier data, it might be a list of dicts
        if isinstance(attr, list):
            for item in attr:
                if isinstance(item, dict):
                    # Look for textLink field
                    if 'textLink' in item:
                        text_link = item['textLink']
                        if text_link and text_link.startswith('mailto:'):
                            emails.append(text_link.replace('mailto:', ''))
        
        # If it's a string that contains JSON
        elif isinstance(attr, str):
            # Try to parse the string
            try:
                parsed = json.loads(attr)
                if isinstance(parsed, list):
                    for item in parsed:
                        if isinstance(item, dict) and 'textLink' in item:
                            text_link = item['textLink']
                            if text_link and text_link.startswith('mailto:'):
                                emails.append(text_link.replace('mailto:', ''))
            except:
                # If parsing fails, maybe the string itself is the email?
                if attr.startswith('mailto:'):
                    emails.append(attr.replace('mailto:', ''))
    
    except Exception as e:
        print(f"Error in row {row.name}: {e}")
    
    return emails

linkedIn_post['extracted_emails'] = linkedIn_post.apply(fix_email_extraction, axis=1)


C:\Users\User\AppData\Local\Temp\ipykernel_6248\3646196855.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linkedIn_post['extracted_emails'] = linkedIn_post.apply(fix_email_extraction, axis=1)


In [162]:
# Function to safely get author fields
def get_author_field(author_data, field):
    if isinstance(author_data, dict):
        return author_data.get(field)
    elif isinstance(author_data, str):
        try:
            parsed = json.loads(author_data)
            return parsed.get(field) if isinstance(parsed, dict) else None
        except:
            return None
    return None

# Extract the fields you want
linkedIn_post['author_name'] = linkedIn_post['author'].apply(lambda x: get_author_field(x, 'name'))
linkedIn_post['author_info'] = linkedIn_post['author'].apply(lambda x: get_author_field(x, 'info'))
linkedIn_post['author_linkedin'] = linkedIn_post['author'].apply(lambda x: get_author_field(x, 'linkedinUrl'))

# Also get public identifier if needed
linkedIn_post['author_public_id'] = linkedIn_post['author'].apply(lambda x: get_author_field(x, 'publicIdentifier'))


C:\Users\User\AppData\Local\Temp\ipykernel_6248\3492286380.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linkedIn_post['author_name'] = linkedIn_post['author'].apply(lambda x: get_author_field(x, 'name'))
C:\Users\User\AppData\Local\Temp\ipykernel_6248\3492286380.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linkedIn_post['author_info'] = linkedIn_post['author'].apply(lambda x: get_author_field(x, 'info'))
C:\Users\User\AppData\Local\Temp\ipykernel_6248\3492286380.py:16: SettingWithCopyWarning

In [163]:
import pandas as pd
import json
from datetime import datetime

# Define a simple helper function
def extract_time(meta):
    # If it's a string, parse it; if it's a dict, use it directly
    data = json.loads(meta) if isinstance(meta, str) else meta
    
    # Get timestamp and convert to readable format
    if data and 'timestamp' in data:
        return datetime.fromtimestamp(data['timestamp'] / 1000).strftime('%Y-%m-%d %H:%M:%S')
    return None

# Create the postedAt column
linkedIn_post['Posted_Time'] = linkedIn_post['postedAt'].apply(extract_time)


C:\Users\User\AppData\Local\Temp\ipykernel_6248\3100386103.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linkedIn_post['Posted_Time'] = linkedIn_post['postedAt'].apply(extract_time)


In [164]:
drop_columns = ['contentAttributes', 'author', 'postedAt']
cleaned_linkedIn_post = linkedIn_post.drop(columns=drop_columns)

In [ ]:
import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

# Define the scopes (permissions) we need to access Google Sheets and Drive
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

# Load the credentials from the JSON file
credentials = Credentials.from_service_account_file("D:\GITHUB\LinkedIn Hiring Posts - Feb 2026(n8n Automation)\credentials.json", scopes=SCOPES)

# Authorize the gspread client using our credentials
client = gspread.authorize(credentials)

# -------------------------------------------------------
# OPEN THE EXISTING SHEET (that you created manually)
# -------------------------------------------------------
# We open by name — make sure this matches exactly what you named your sheet
# spreadsheet = client.open("LinkedIn Posts Data")


# Option 2: Open by just the ID (the long string between /d/ and /edit)
spreadsheet = client.open_by_key("1_3BJkGRZHEvq0XkfJGOB3lCe8rJgeGG5JW7vnY_uxxw")

# Points to your new tab, not Sheet1
worksheet = spreadsheet.worksheet("LinkedIn Posts Data")


# -------------------------------------------------------
# PREPARE THE DATAFRAME FOR UPLOAD
# -------------------------------------------------------
# Extract column headers from the dataframe
headers = cleaned_linkedIn_post.columns.tolist()

# Convert all rows to list of lists (str() avoids issues with NaN, dates, etc.)
rows = cleaned_linkedIn_post.astype(str).values.tolist()

# Combine headers + rows into one complete dataset
data = [headers] + rows

# -------------------------------------------------------
# CLEAR OLD DATA & PUSH NEW DATA TO GOOGLE SHEET
# -------------------------------------------------------
# Clear the sheet first to avoid leftover data from previous runs
worksheet.clear()

# Write all data starting from cell A1
worksheet.update("A1", data)

print("✅ Data successfully copied to Google Sheets!")
print(f"🔗 Open it here: {spreadsheet.url}")

C:\Users\User\AppData\Local\Temp\ipykernel_6248\2828831885.py:50: DeprecationWarning: [Deprecated][in version 6.0.0]: Method signature's arguments 'range_name' and 'values' will change their order. We recommend using named arguments for minimal impact. In addition, the argument 'values' will be mandatory of type: 'List[List]'. (ex) Worksheet.update(values = [[]], range_name=) 
  worksheet.update("A1", data)


✅ Data successfully copied to Google Sheets!
🔗 Open it here: https://docs.google.com/spreadsheets/d/1_3BJkGRZHEvq0XkfJGOB3lCe8rJgeGG5JW7vnY_uxxw
